In [1]:
#pip install matplotlib notebook opencv-contrib-python opencv-python opencv-stubs
import cv2
import numpy as np

In [2]:
def nothing(_):
    pass

In [3]:
def get_params(win):
    h_low = cv2.getTrackbarPos("1) Hue low (0-179)", win)
    h_high = cv2.getTrackbarPos("2) Hue high (0-179)", win)
    s_low = cv2.getTrackbarPos("3) Sat low (0-255)", win)
    s_high = cv2.getTrackbarPos("4) Sat high (0-255)", win)
    v_low = cv2.getTrackbarPos("5) Val low (0-255)", win)
    v_high = cv2.getTrackbarPos("6) Val high (0-255)", win)
    median_k = cv2.getTrackbarPos("7) Median k (odd)", win)
    morph_k = cv2.getTrackbarPos("8) Morph k", win)
    morph_it = cv2.getTrackbarPos("9) Morph iters", win)
    min_area_pct = cv2.getTrackbarPos("10) Min area %", win)

    median_k = max(1, median_k | 1)
    morph_k = max(1, morph_k)
    morph_it = max(0, morph_it)
    s_high = max(s_low, s_high)
    v_high = max(v_low, v_high)

    return (
        h_low,
        h_high,
        s_low,
        s_high,
        v_low,
        v_high,
        median_k,
        morph_k,
        morph_it,
        min_area_pct,
    )

In [4]:
def hsv_range(hsv, params):
    h_low, h_high, s_low, s_high, v_low, v_high, *_ = params
    if h_low <= h_high:
        lower = np.array([h_low, s_low, v_low], dtype=np.uint8)
        upper = np.array([h_high, s_high, v_high], dtype=np.uint8)
        mask = cv2.inRange(hsv, lower, upper)
    else:
        lower1 = np.array([h_low, s_low, v_low], dtype=np.uint8)
        upper1 = np.array([179, s_high, v_high], dtype=np.uint8)
        lower2 = np.array([0, s_low, v_low], dtype=np.uint8)
        upper2 = np.array([h_high, s_high, v_high], dtype=np.uint8)
        mask = cv2.bitwise_or(
            cv2.inRange(hsv, lower1, upper1),
            cv2.inRange(hsv, lower2, upper2),
        )
    return mask

In [5]:
def hsv_bitwise(img, mask):
    return cv2.bitwise_and(img, img, mask=mask)


In [6]:
def hsv_median(img, mask, k):
    out = cv2.bitwise_and(img, img, mask=mask)
    return cv2.medianBlur(out, k)

In [7]:
def morphology(mask, k, iters=1):
    kernel = np.ones((k, k), np.uint8)
    # morph = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=iters)
    morph = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iters)
    return morph

In [8]:
def hsv_range_multiple(hsv, params_list):
    mask = None
    for params in params_list:
        m = hsv_range(hsv, params)
        mask = m if mask is None else cv2.bitwise_or(mask, m)
    return mask

In [9]:
def draw_marker(img, mask, min_area_px):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img
    contours = [c for c in contours if cv2.contourArea(c) >= min_area_px]
    if not contours:
        return img
    
    out = img.copy()

    c = max(contours, key=cv2.contourArea)

    for c in contours:
        M = cv2.moments(c)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        cv2.drawMarker(
            out,
            (cx, cy),
            (0, 255, 0),
            markerType=cv2.MARKER_CROSS,
            markerSize=20,
            thickness=2,
        )
        cv2.putText(
            out,
            "object",
            (cx - 60, cy + 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

    return out

In [14]:
def main():
    win_ctrl = "Controls"
    win_orig = "original"
    win_mask = "mask"
    win_bitw = "bitwise"
    win_med = "median"
    win_morph = "morphology"
    win_mark = "marker"

    image = cv2.imread("assets/test.png")
    if image is None:
        raise RuntimeError("Nie znaleziono pliku assets/test.png")
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    H, W = image.shape[:2]

    cv2.namedWindow(win_ctrl, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_orig, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_mask, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_bitw, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_med, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_morph, cv2.WINDOW_NORMAL)
    cv2.namedWindow(win_mark, cv2.WINDOW_NORMAL)

    cv2.createTrackbar("1) Hue low (0-179)", win_ctrl, 0, 179, nothing)
    cv2.createTrackbar("2) Hue high (0-179)", win_ctrl, 179, 179, nothing)
    cv2.createTrackbar("3) Sat low (0-255)", win_ctrl, 50, 255, nothing)
    cv2.createTrackbar("4) Sat high (0-255)", win_ctrl, 255, 255, nothing)
    cv2.createTrackbar("5) Val low (0-255)", win_ctrl, 50, 255, nothing)
    cv2.createTrackbar("6) Val high (0-255)", win_ctrl, 255, 255, nothing)
    cv2.createTrackbar("7) Median k (odd)", win_ctrl, 3, 21, nothing)
    cv2.createTrackbar("8) Morph k", win_ctrl, 3, 21, nothing)
    cv2.createTrackbar("9) Morph iters", win_ctrl, 1, 10, nothing)
    cv2.createTrackbar("10) Min area %", win_ctrl, 1, 100, nothing)

    ctrl_pad = np.zeros((40, 600, 3), dtype=np.uint8)
    cv2.imshow(win_ctrl, ctrl_pad)

    while True:
        params = get_params(win_ctrl)
        mask = hsv_range(hsv, params)
        bitwise = hsv_bitwise(image, mask)
        median = hsv_median(bitwise, mask, params[6])
        morph = morphology(mask, params[7], params[8])
        min_area_px = 0.01
        marker_img = draw_marker(image, morph, min_area_px)

        cv2.imshow(win_orig, image)
        cv2.imshow(win_mask, mask)
        cv2.imshow(win_bitw, bitwise)
        cv2.imshow(win_med, median)
        cv2.imshow(win_morph, morph)
        cv2.imshow(win_mark, marker_img)

        if cv2.waitKey(30) == 27:
            break
        elif cv2.waitKey(30) == ord("p"):
            print(get_params(win_ctrl))

    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

In [23]:
def video_main():
    vid = cv2.VideoCapture("assets/moving_ball2.mp4")
    width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = vid.get(cv2.CAP_PROP_FPS)
    out = cv2.VideoWriter(
        "assets/moving_ball2_new.mp4",
        cv2.CAP_ANY,
        fps,
        (width, height),
    )
    #params = (0,179,120,255,130,255,3,5,1,0.01)
    params_red = (0,10,165,255,120,255,3,5,3,0.012) #red git
    params_green = (40,60,80,255,110,255,3,5,3,0.012) #green git
    params_yellow = (20,60,110,255,120,255,3,5,3,0.012) #yellow git
    params_blue = (60,130,65,255,70,255,3,5,3,0.012) #blue git

    params_list = [params_red, params_green, params_yellow, params_blue]

    current_frame = 0
    while True:
        ret, frame = vid.read()
        if not ret:
            print("End?")
            break
        current_frame += 1
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        """mask = hsv_range(hsv,params)
        morph = morphology(mask, params[7], params[8])
        min_area_px = int((params[9] / 100.0) * (height * width))
        """
        mask = hsv_range_multiple(hsv, params_list)
        morph = morphology(mask, params_red[7], params_red[8])
        min_area_px = int((params_red[9] / 100.0) * (height * width))
        new_frame = draw_marker(frame, morph, min_area_px)
        out.write(new_frame)
        if current_frame % 10 == 0:
            print(f"{current_frame / fps:.2f}")

    vid.release()
    out.release()

if __name__ == "__main__":
    video_main()

0.33
0.67
1.00
1.33
1.67
2.00
2.33
2.67
3.00
3.33
3.67
4.00
4.33
4.67
5.00
5.33
5.67
6.00
6.33
6.67
7.00
7.33
7.67
8.00
8.33
8.67
9.00
9.33
9.67
10.00
10.33
10.67
11.00
11.33
11.67
12.00
12.33
12.67
13.00
13.33
13.67
14.00
14.33
14.67
15.00
15.33
15.67
16.00
16.33
16.67
17.00
17.33
17.67
18.00
18.33
18.67
19.00
19.33
19.67
20.00
20.33
20.67
21.00
21.33
21.67
22.00
22.33
22.67
23.00
23.33
23.67
24.00
24.33
24.67
25.00
25.33
25.67
26.00
26.33
26.67
27.00
27.33
27.67
28.00
28.33
28.67
29.00
29.33
29.67
30.00
30.33
30.67
31.00
31.33
31.67
32.00
32.33
32.67
33.00
33.33
33.67
34.00
34.33
34.67
35.00
35.33
35.67
36.00
36.33
36.67
37.00
37.33
37.67
38.00
38.33
38.67
39.00
39.33
39.67
40.00
40.33
40.67
41.00
41.33
41.67
42.00
42.33
42.67
43.00
43.33
43.67
44.00
44.33
44.67
45.00
45.33
45.67
46.00
46.33
46.67
47.00
47.33
47.67
48.00
48.33
48.67
49.00
49.33
49.67
50.00
50.33
50.67
51.00
51.33
51.67
52.00
52.33
52.67
53.00
53.33
53.67
54.00
54.33
54.67
55.00
55.33
55.67
56.00
56.33
56.67
57.00
57.